> **Generated notebook — do not edit here.**  
> Source: `01_scripts/02_differential_expression_analysis.Rmd`, which is also a chapter of the course book.  
> To change anything, edit the Rmd and run `python3 util/rmd_to_ipynb.py`.
>
> Run the notebooks in order — **01 → 02 → 03** — with the **R** kernel; each step saves results that the next one loads.
>
> **Pick ONE dataset** (*S. aureus* or human) and work through it properly — if your group finishes early, start on the other one.

**First time here? Four things to expect:**

- 🌐 **Browser:** use **Chrome, Firefox or Edge** — Codespaces does not work reliably in Safari.
- 🧮 **Kernel:** when VS Code asks you to *Select Kernel*, choose **Jupyter Kernel...** → **R**. The notebooks run R, not Python.
- ⚠️ **"No text editor active" pop-up:** a harmless warning from the R extension — your code still runs. Just close it.
- ▶️ **Running cells:** use **Shift+Enter** or the ▶ button next to the cell — **not Ctrl+Enter**, which the R extension intercepts. The first cell can take a moment while the R kernel starts.

In [ ]:
# Match the report's defaults: warnings hidden (warning=FALSE in the Rmd)
# and 7 x 5 inch figures. Remove the warn option to see warnings.
options(warn = -1, repr.plot.width = 7, repr.plot.height = 5)

# Make tables display in Jupyter the way they do in the rendered report.
# kable()/kableExtra return HTML that the R kernel would otherwise show as
# raw text; DT::datatable() is an interactive widget whose JavaScript does
# not run in the VS Code output pane, so it is shown as a static table.
options(knitr.table.format = "html")
local({
  css <- paste0("<style>table.table,table.dataframe{border-collapse:collapse;font-size:0.9em}",
                ".table th,.table td{padding:3px 10px;border-bottom:1px solid #ddd}",
                ".table-striped tbody tr:nth-child(odd){background:#f5f7fa}</style>")
  registerS3method("repr_html", "knitr_kable", function(obj, ...) {
    paste0(css, paste(obj, collapse = "\n"))
  }, envir = asNamespace("repr"))
  registerS3method("repr_html", "datatables", function(obj, ...) {
    d <- as.data.frame(obj$x$data, stringsAsFactors = FALSE, check.names = FALSE)
    note <- if (nrow(d) > 100) sprintf(
      "<p style='font-size:0.85em;color:#666'><em>Static preview: first 100 of %d rows.</em></p>", nrow(d)) else ""
    tbl <- knitr::kable(head(d, 100), format = "html", row.names = FALSE,
                        table.attr = "class='table table-striped'")
    paste0(css, note, paste(tbl, collapse = "\n"))
  }, envir = asNamespace("repr"))
})

# Background

This report covers differential expression (DE) analysis of bulk RNA-seq data from primary human airway smooth muscle (ASM) cells treated with dexamethasone ([Himes *et al.*, 2014](https://doi.org/10.1371/journal.pone.0099625)) using `DESeq2`.

We test one contrast:

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

- **Dexamethasone (treatment) vs untreated (control)** — which genes change expression in response to the glucocorticoid?

</div>

<div style="background:#eaf4fd;border-left:5px solid #3498db;padding:0.5em 1em;margin:1em 0;border-radius:6px;">

<strong>⭐ Important:</strong> This script assumes you have already run `01_quality_control.Rmd` and that the `dds` object has been saved. If not, re-run the QC script first.

</div>

<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:0.5em 1em;margin:1em 0;">

<strong>🎓 Teaching design.</strong> Each donor contributes a treated and an untreated sample, so donor
is a paired factor. We run two models:
<ul>
<li><strong>Paired (recommended):</strong> <code>~ donor + condition</code> — adjusts for donor baseline before testing treatment.</li>
<li><strong>Simple (for comparison):</strong> <code>~ condition</code> — ignores donor.</li>
</ul>
Comparing them shows how accounting for a known confounder changes power and the gene list.
The paired model is used for all downstream analysis.

</div>

**Setup the Environment**

In [ ]:
library(tidyverse)
library(DESeq2)
library(ggpubr)
library(pheatmap)
library(RColorBrewer)
library(knitr)
library(kableExtra)
library(DT)

## Differential Expression Analysis

<div style="background:#f8d7da;border-left:4px solid #721c24;padding:10px;margin:10px 0;">

  <strong>⚠️ Warning:</strong> Always use raw integer counts as input. Do not use TPM, FPKM, or any normalised values — DESeq2 handles normalisation internally.

</div>

`DESeq2` fits a negative binomial model to the raw counts and performs a Wald test for each gene. Internally it runs three steps in sequence: size factor estimation (normalisation), dispersion estimation, and the Wald test. These can also be run separately, but `DESeq()` handles all three in one call.

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

**What DESeq2 does internally — three steps:**

**1. Size factor estimation (normalisation)**

Each sample gets a size factor that accounts for differences in sequencing depth.
DESeq2 uses the **median-of-ratios** method: for each gene, it computes the ratio
of each sample's count to the geometric mean count across all samples, then takes
the median ratio across all genes as the size factor. This is more robust than
simply dividing by total counts because it is not affected by a small number of
highly expressed genes dominating the library.

**2. Dispersion estimation**

RNA-seq count data is overdispersed — the variance is greater than the mean
(unlike a Poisson distribution which assumes variance = mean). DESeq2 models
this with a **negative binomial distribution** and estimates a dispersion
parameter per gene, which describes how much counts vary between replicates
beyond what is expected by chance. Genes with few replicates get their
dispersion estimates shrunk toward a fitted trend across all genes
(**empirical Bayes shrinkage**) — this stabilises estimates for genes with
low counts.

**3. Wald test**

For each gene, DESeq2 fits the negative binomial model and tests whether the
log2 fold change between conditions is significantly different from zero using
a **Wald test**. The resulting p-values are then corrected for multiple testing
using the **Benjamini-Hochberg** procedure to control the false discovery rate (FDR).

---

**Why does the order matter?**

With the design `~ donor + condition`, the last term is the one tested. `resultsNames(dds)`
will list `condition_treatment_vs_control` — the fold change is calculated as **treatment / control**:

- **Positive LFC** → gene is upregulated in dexamethasone-treated cells
- **Negative LFC** → gene is downregulated in dexamethasone-treated cells

If control were not set as the reference level, the fold change would be
inverted and your biological interpretation would be backwards.

</div>

In [ ]:
git_root <- system("git rev-parse --show-toplevel", intern = TRUE)

dds <- readRDS(file.path(git_root, "results", "human", "rds", "dds_human_asm.rds"))

cat("✅ DDS loaded\n")
cat("   Dimensions  :", nrow(dds), "genes ×", ncol(dds), "samples\n")
cat("   Design      :", paste(deparse(design(dds)), collapse = ""), "\n")
cat("   Conditions  :", paste(levels(dds$condition), collapse = " vs "), "\n")

<div style="background:#d1ecf1;border-left:4px solid #0c5460;padding:10px;margin:10px 0;">

  <strong>📌 Remember:</strong> The variable of interest is at the end of the design formula, and the control group is the first (reference) factor level. Both were set in the QC script.

</div>

**Run DESeq2 (paired design)**

In [ ]:
dds <- DESeq(dds)
resultsNames(dds)

## Teaching Comparison — Paired vs Simple Design

Before extracting the final results, we fit the simple `~ condition` model on the same data and compare how many genes each design calls significant. This makes the effect of accounting for donor concrete.

In [ ]:
# Simple model: drop the donor term
dds_simple <- dds
design(dds_simple) <- ~ condition
dds_simple <- DESeq(dds_simple)

res_paired_tmp <- results(dds,
                          contrast = c("condition", "treatment", "control"),
                          alpha    = 0.05)
res_simple_tmp <- results(dds_simple,
                          contrast = c("condition", "treatment", "control"),
                          alpha    = 0.05)

n_sig <- function(r) sum(r$padj < 0.05, na.rm = TRUE)

comparison_tbl <- data.frame(
  Design            = c("~ donor + condition (paired)", "~ condition (simple)"),
  `Significant genes (padj < 0.05)` = c(n_sig(res_paired_tmp), n_sig(res_simple_tmp)),
  check.names = FALSE
)

kable(comparison_tbl, caption = "Effect of design on the number of significant genes") %>%
  kable_styling(bootstrap_options = c("striped", "hover"), full_width = FALSE)

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

**How to read this comparison.** Pairing on donor removes the between-donor baseline variation from
the residual, which usually *lowers* the per-gene noise and *increases* power to detect the treatment
effect. If the paired model calls more genes significant here, that is the mechanism. If the two are
similar, donor happened to explain little variance in this subset — but the paired model is still the
correct specification given the experimental design, so we keep it.

</div>

<div style="background:#eaf4fd;border-left:5px solid #3498db;padding:0.5em 1em;margin:1em 0;border-radius:6px;">

**What we see in this dataset.** The paired model (`~ donor + condition`) calls roughly **3,800**
genes significant at padj < 0.05, versus about **2,600** for the simple model (`~ condition`) — an
increase of around 1,200 genes (~50%). This is a large, concrete demonstration of the paired-design
benefit: the four donors differ substantially in their baseline expression, and once that donor
variation is accounted for, the treatment signal becomes much easier to detect. This is exactly the
inter-patient variability the study design anticipated, and it is why we use the paired model for all
downstream steps.

</div>

## 🧠 Interpretation questions

<div style="background:#f2efff;border-left:5px solid #6c5ce7;border-radius:6px;padding:10px 16px;margin:10px 0;">

1. Your table shows the paired model (`~ donor + condition`) finds over a thousand more significant genes than the simple model — same data, same treatment. What changed? Where did the donor-to-donor differences "go" in the simple model, and why does giving them a name in the paired model make the treatment effect easier to see? Was your prediction from the QC notebook right?

</div>

## Extracting Results

`results()` extracts the Wald test output for a given contrast. By default DESeq2 applies **independent filtering** — it removes genes with very low mean counts before multiple testing correction, which increases power to detect DE genes at a given FDR threshold.

We set `alpha = 0.05` to optimise filtering for a 5% FDR cutoff.

In [ ]:
res <- results(dds,
               contrast = c("condition", "treatment", "control"),
               alpha    = 0.05)

summary(res)

<div style="background:#eaf4fd;border-left:5px solid #3498db;padding:0.5em 1em;margin:1em 0;border-radius:6px;">

**Reading the summary.** Of 14,080 tested genes, about **2,060 are up** and **1,770 are down** at
padj < 0.05 (roughly 15% and 13%). Two lines are worth noting: **outliers = 0** means no gene was
flagged by Cook's distance as driven by a single extreme sample, and **low counts = 0** means
independent filtering did not need to remove additional low-expression genes (the pre-filtering in
script 01 already handled them). A large but balanced set of up- and down-regulated genes is the
expected signature of a strong, genome-wide treatment response — dexamethasone is a potent
transcriptional regulator, so this magnitude is biologically plausible.

</div>

The results table contains one row per gene with the following columns:

| Column | Description |
|---|---|
| `baseMean` | Mean normalised count across all samples |
| `log2FoldChange` | log2(treatment / control) |
| `lfcSE` | Standard error of the log2FC estimate |
| `stat` | Wald statistic |
| `pvalue` | Raw p-value |
| `padj` | Benjamini–Hochberg adjusted p-value |

---

## Log2 Fold Change Shrinkage

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

**What is this step and why do we need it?**

 After running DESeq2, every gene gets a raw log2 fold change (LFC) estimate.
 For **lowly expressed genes**, these estimates are unreliable — a gene with
 1 count in control and 3 in treatment looks like a 3-fold change, but this
 is just noise from low counts.

 `lfcShrink()` corrects this using **empirical Bayes shrinkage**: it pulls
 noisy estimates (low counts, high uncertainty) toward zero, while leaving
 well-supported fold changes (high counts, consistent signal) largely
 unchanged. Think of it as a confidence-weighted adjustment.
<div style="background:#fdf2e9;border-left:5px solid #e67e22;border-radius:4px;padding:10px 16px;margin:10px 0;">

  <strong>⭐ Important:</strong> shrinkage does **not** change your p-values or which genes
 are significant — those come from the `results()` call above. Shrunken LFCs
 are used exclusively for **ranking and visualisation** (volcano plot, heatmap).

</div>

 **Why `apeglm`?** Three shrinkage methods exist in DESeq2:
 - `apeglm` — most accurate for simple two-group comparisons like ours ✅
 - `ashr` — more flexible; needed when comparing two non-reference groups
 - `normal` — the original method, now outdated and not recommended

 For a treatment vs control design, `apeglm` is the right choice.
 See [Zhu *et al.*, 2019](https://doi.org/10.1093/bioinformatics/btz585) for benchmarks.

</div>

In [ ]:
res_shrunk <- lfcShrink(dds,
                        coef  = "condition_treatment_vs_control",
                        type  = "apeglm")

summary(res_shrunk, alpha = 0.05)

---

## Sorting and Filtering Significant Genes

We order genes by adjusted p-value and filter to retain only those passing our significance thresholds: **padj < 0.05** and **|log2FC| ≥ 1** (i.e., at least a 2-fold change).

In [ ]:
res_df <- as.data.frame(res_shrunk) %>%
  rownames_to_column("gene") %>%
  arrange(padj) %>%
  filter(!is.na(padj))

res_sig <- res_df %>%
  filter(padj < 0.05, abs(log2FoldChange) >= 1)

cat("Total genes tested        :", nrow(res_df), "\n")
cat("Significant genes         :", nrow(res_sig), "\n")
cat("  Upregulated (LFC ≥ 1)   :", sum(res_sig$log2FoldChange >= 1), "\n")
cat("  Downregulated (LFC ≤ -1):", sum(res_sig$log2FoldChange <= -1), "\n")

<div style="background:#eaf4fd;border-left:5px solid #3498db;padding:0.5em 1em;margin:1em 0;border-radius:6px;">

**What the fold-change filter does.** The `summary()` above counted ~3,800 genes at padj < 0.05 alone.
Adding the **|log2FC| ≥ 1** requirement (at least a 2-fold change) narrows this to about **700** genes
(~380 up, ~320 down). The difference is the large group of genes that are statistically significant but
only modestly changed — real, but small in magnitude. Filtering on effect size as well as significance
focuses the downstream analysis on the genes most likely to be biologically meaningful. The up/down
split being roughly balanced is consistent with a broad regulatory response rather than a one-directional
shift.

</div>

**Biological sanity check — known dexamethasone-responsive genes**

Himes *et al.* highlighted several well-established glucocorticoid-responsive genes. We check where they
land in our results rather than assuming — if the pipeline is working, these should be significant and in
the expected direction (dexamethasone typically *induces* these).

In [ ]:
known_genes <- c("DUSP1", "KLF15", "PER1", "TSC22D3", "CRISPLD2", "FKBP5", "ZBTB16")

res_df %>%
  filter(gene %in% known_genes) %>%
  select(gene, log2FoldChange, padj) %>%
  mutate(across(where(is.numeric), \(x) signif(x, 3))) %>%
  arrange(padj) %>%
  kable(caption = "Known glucocorticoid-responsive genes in our results") %>%
  kable_styling(bootstrap_options = c("striped", "hover"), full_width = FALSE)

<div style="background:#d4edda;border-left:4px solid #28a745;padding:0.5em 1em;margin:1em 0;border-radius:6px;">

**How to read this.** If these canonical genes appear with positive log2FC and small padj, it is strong
evidence the analysis is capturing real glucocorticoid biology, not noise. A gene missing from the table
was filtered out earlier for low counts (not expressed in these cells); that is not a problem in itself.
In this dataset all of the recovered markers are strongly **induced** (positive log2FC) and highly
significant — including FKBP5, TSC22D3 (GILZ) and ZBTB16, three of the most reliable glucocorticoid-response
genes, and CRISPLD2, the gene the original study is named after. This is the expected positive-control
result and confirms the pipeline is working correctly.

</div>

<div style="background:#eaf4fd;border-left:5px solid #3498db;border-radius:4px;padding:10px 16px;margin:10px 0;">

  <strong>📘 Note:</strong> Known dexamethasone-responsive genes reported by Himes <em>et al.</em>
  include <em>DUSP1</em>, <em>KLF15</em>, <em>PER1</em>, <em>TSC22D3</em> and <em>CRISPLD2</em>.
  Checking whether these appear among your top hits is a good biological sanity check —
  but let the analysis produce them rather than assuming their rank.

</div>

---

**Interactive Results Table**

In [ ]:
DT::datatable(
  res_sig %>% mutate(across(where(is.numeric), \(x) round(x, 4))),
  rownames   = FALSE,
  extensions = c('Buttons', 'Scroller'),
  options    = list(
    dom         = 'Bfrtip',
    buttons     = c('copy', 'csv'),
    scrollX     = TRUE,
    scrollY     = 300,
    scroller    = TRUE
  ),
  caption = 'Significant DE genes — dexamethasone vs untreated (padj < 0.05, |LFC| ≥ 1)'
)

## 🧠 Interpretation questions

<div style="background:#f2efff;border-left:5px solid #6c5ce7;border-radius:6px;padding:10px 16px;margin:10px 0;">

2. Dexamethasone works by binding the glucocorticoid receptor, made by the gene *NR3C1*. Before looking: predict where *NR3C1* sits in your results — top hit, modest, or absent? Now check your tables (remember the interactive table only shows genes passing both cutoffs). If the protein driving this whole response barely moves in the mRNA data, what does that tell you about what a count can measure?

</div>

## Visualisation

**Volcano Plot**

A volcano plot combines fold change (x-axis) and statistical significance (y-axis) into a single view. Genes in the upper corners are both strongly and significantly changed — these are the most biologically interesting candidates.

The dashed lines mark our thresholds: |log2FC| = 1 and padj = 0.05.

In [ ]:
res_df <- res_df %>%
  mutate(
    significance = case_when(
      padj < 0.05 & log2FoldChange >=  1 ~ "Up",
      padj < 0.05 & log2FoldChange <= -1 ~ "Down",
      TRUE                               ~ "NS"
    )
  )

top_labels <- res_df %>%
  filter(significance != "NS") %>%
  slice_min(padj, n = 10)

volcano <- ggplot(res_df, aes(x = log2FoldChange,
                              y = -log10(padj),
                              color = significance)) +
  geom_point(alpha = 0.7, size = 1.8) +
  geom_vline(xintercept = c(-1, 1),  linetype = "dashed", color = "black") +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "black") +
  geom_text(data = top_labels,
            aes(label = gene),
            size = 2.8, vjust = -0.6, show.legend = FALSE) +
  scale_color_manual(values = c("Up" = "red", "Down" = "blue", "NS" = "grey70")) +
  theme_pubr(border = TRUE) +
  labs(
    title = "Volcano Plot — Dexamethasone vs Untreated",
    x     = "log2(Fold Change)",
    y     = "-log10(adjusted p-value)",
    color = NULL
  )

volcano

## MA Plot

An MA plot shows log2 fold change (y-axis) against mean expression (x-axis). It helps assess whether the fold change estimates are stable across the expression range. After shrinkage, estimates for lowly expressed genes (left side) should be pulled toward zero.

In [ ]:
plotMA(res_shrunk,
       alpha = 0.05,
       main  = "MA Plot — Dexamethasone vs Untreated (shrunken LFC)",
       ylab  = "log2 Fold Change")
abline(h = c(-1, 1), lty = 2, col = "black")

## Top Gene Count Plot

Plotting raw normalised counts for the most significant gene provides a sanity check: the direction and magnitude of the fold change should be visually obvious. If it isn't, investigate further.

In [ ]:
top_gene <- res_sig$gene[1]

plotCounts(dds,
           gene     = top_gene,
           intgroup = "condition",
           main     = paste("Normalised counts —", top_gene))

## Heatmap of Significant Genes

Clustering significant DE genes across samples provides a gene-level view of the expression differences. Genes should show clear block structure: high expression in treatment and low in control (or vice versa). With the paired design, you can also annotate donor to see whether the treatment effect is consistent across donors.

In [ ]:
vsd <- varianceStabilizingTransformation(dds, blind = FALSE)

sig_genes <- res_sig %>%
  slice_min(padj, n = min(50, nrow(res_sig))) %>%
  pull(gene)

df_anno    <- as.data.frame(colData(vsd)[, c("condition", "donor"), drop = FALSE])

pheatmap(
  assay(vsd)[sig_genes, ],
  cluster_cols             = TRUE,
  cluster_rows             = TRUE,
  scale                    = "row",
  clustering_distance_rows = "euclidean",
  clustering_distance_cols = "euclidean",
  annotation_col           = df_anno,
  show_colnames            = TRUE,
  show_rownames            = TRUE,
  fontsize_row             = 7,
  main                     = "Top DE genes — Dexamethasone vs Untreated (VST, row-scaled)"
)

## Saving Results

In [ ]:
results_dir <- file.path(git_root, "results", "human")
dir.create(results_dir, showWarnings = FALSE)

write.table(
  res_df,
  file      = file.path(results_dir, "DE_treatment_vs_control_all.tsv"),
  sep       = "\t",
  quote     = FALSE,
  row.names = FALSE
)

write.table(
  res_sig,
  file      = file.path(results_dir, "DE_treatment_vs_control_significant.tsv"),
  sep       = "\t",
  quote     = FALSE,
  row.names = FALSE
)

cat("Results saved to:", results_dir, "\n")

---

## Summary

| Step | Decision made |
|---|---|
| Design formula | `~ donor + condition` (paired); `~ condition` shown for comparison |
| Contrast | dexamethasone (treatment) vs untreated (control) |
| LFC shrinkage | `apeglm` method |
| Significance threshold | padj < 0.05 |
| Fold change threshold | \|log2FC\| ≥ 1 (2-fold) |
| Enrichment (next script) | g:Profiler via `gprofiler2` (human) |

Significant genes are saved and ready for gene set enrichment analysis in the next script.

</br>

In [ ]:
sessionInfo()